In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats

import climattr as eea

plt.rcParams.update({'font.size': 14})

In [ ]:
data = {'mean_2m_air_temp_degree1': [], 'ens': []}
for i in range(525):
    try:
        ext = pd.read_csv(f'~/cpdn_nonnerc/aaim/dengue/predict-all/ext-ens{i:03d}.csv')
        ext = ext[ext['region'].isin(['Middle-West', 'Southeast', 'South', 'North', 'Northeast'])]
        ext = (ext['mean_2m_air_temp_degree1'] * ext['population']).sum() / ext['population'].sum()
        data['mean_2m_air_temp_degree1'].append(ext)
        data['ens'].append(i)
    except Exception as e:
        print(i)
        print(e)

ext = pd.DataFrame(data)

data = {'mean_2m_air_temp_degree1': [], 'ens': []}
for i in range(525):
    try:
        nat = pd.read_csv(f'~/cpdn_nonnerc/aaim/dengue/predict-all/ext-nat-ens{i:03d}.csv')
        nat = nat[nat['region'].isin(['Middle-West', 'Southeast', 'South', 'North', 'Northeast'])]
        nat = (nat['mean_2m_air_temp_degree1'] * nat['population']).sum() / nat['population'].sum()
        data['mean_2m_air_temp_degree1'].append(nat)
        data['ens'].append(i)
    except Exception as e:
        print(i)
        print(e)

nat = pd.DataFrame(data)

In [ ]:
obs_temp = pd.read_csv('data/model_input_brazil_immunity_city.csv', parse_dates=['date_first_symptoms'])
obs_temp = obs_temp[(obs_temp['date_first_symptoms'] >= '2023-08-01') & (obs_temp['date_first_symptoms'] <= '2024-07-31')]
obs_temp = (obs_temp['mean_2m_air_temp_degree1'] * obs_temp['population']).sum() / obs_temp['population'].sum()

In [ ]:
eea.attribution.attribution_metrics(
    ext.set_index('ens').to_xarray()['mean_2m_air_temp_degree1'],
    nat.set_index('ens').to_xarray()['mean_2m_air_temp_degree1'],
    stats.norm,
    obs_temp
)

In [ ]:
fig, [ax1, ax2] = plt.subplots(1, 2, figsize=(12,5))

eea.attribution.histogram_plot(
    ax1, 
    ext.set_index('ens').to_xarray()['mean_2m_air_temp_degree1'],
    nat.set_index('ens').to_xarray()['mean_2m_air_temp_degree1'],
    stats.norm,
    obs_temp
)

eea.attribution.rp_plot(
    ax2, 
    ext.set_index('ens').to_xarray()['mean_2m_air_temp_degree1'],
    nat.set_index('ens').to_xarray()['mean_2m_air_temp_degree1'],
    stats.norm,
    obs_temp
)

for ax in [ax1, ax2]:
    legend = ax.legend()
    legend.remove()

ax1.set_ylabel('PDF')
ax1.set_xlabel(r'Temperature ($^o$C)')

ax2.set_xlabel('Return Period (years)')
ax2.set_ylabel(r'Temperature ($^o$C)')



# ---- LEGEND ----
legend_elements = [
    Patch(facecolor='C0', label='NAT', alpha=0.5),
    Patch(facecolor='C1', label='ACT', alpha=0.5),
]
ax1.legend(handles=legend_elements, frameon=False)

# ---- LEGEND ----
legend_elements = [
    Line2D([0], [0], color='C0', marker='o', linestyle='none', label='NAT'),
    Line2D([0], [0], color='C1', marker='o', linestyle='none', label='ACT'),
]
ax2.legend(handles=legend_elements, frameon=False)

plt.tight_layout()
plt.savefig('img/figures10_attribution_signal.pdf', dpi=300, bbox_inches='tight')